# Time-series analysis — QuPath measurements

**Template** — edit the *Configuration* cell and run all cells.

Input: QuPath measurement TSV (standard or fiji-tools filename convention) + `plate_map.csv`.  
Output: QC object-count chart and per-condition time-series with 95 % CI across biological replicates.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path().resolve().parents[1]  # repo root; adjust if notebook is not in notebooks/<subdir>/
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import incucyte as ic

## Configuration

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
RESULTS_TSV   = ROOT / "data" / "TODO_experiment" / "Results" / "measurements.tsv"
PLATE_MAP_CSV = ROOT / "data" / "TODO_experiment" / "plate_map.csv"
PLATE_IDS     = None          # multi-plate: ["VID9955", "VID9960"]; None for single-plate

# Time index parameters — required when filenames have no embedded timestamp
# (fiji-tools stack exports). Set INTERVAL_MIN=None if timestamps are in filenames.
INTERVAL_MIN  = 30            # minutes between consecutive QuPath time indices
START_MIN     = 0             # elapsed_min for time index 0 (e.g. 1440 = 24 h offset)

CLASSIFICATION = None         # keep only one class, e.g. "Neurosphere"; None keeps all
METRICS = ["Circularity", "Area px^2"]  # measurement columns to plot
# ────────────────────────────────────────────────────────────────────────────

## 1. Load data

In [ ]:
df = ic.load_qupath(RESULTS_TSV, interval_min=INTERVAL_MIN, start_min=START_MIN)
plate_map = ic.load_plate_map(PLATE_MAP_CSV, plate_ids=PLATE_IDS)
df = ic.merge_plate_map(df, plate_map)

if CLASSIFICATION is not None:
    df = ic.filter_by(df, Classification=CLASSIFICATION)

print(f"{len(df):,} objects  |  {df['Well'].nunique()} wells  |  {df['elapsed_min'].nunique()} timepoints")
df.head(3)

## 2. QC — object counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ic.qc_counts(df, groupby="Well", ax=ax)
ax.set_title("Total object count per well (summed across all timepoints)")
plt.tight_layout()

## 3. Aggregate

Two steps:
1. `to_well_means` — per-object → per-well mean at each timepoint  
2. Group across technical replicates → one value per biological replicate (sample × condition × timepoint). The CI in the time-series plot represents variability across these biological replicates.

In [ ]:
df_well = ic.to_well_means(df, metrics=METRICS)

# Average technical replicates within each (sample, condition, timepoint).
# Adjust the groupby columns if your plate map uses different column names.
df_bio = (
    df_well
    .groupby(["sample", "condition", "elapsed_min"])[METRICS]
    .mean()
    .reset_index()
)
df_bio["elapsed_h"] = df_bio["elapsed_min"] / 60

print(f"Biological replicates per condition:")
print(df_bio.groupby("condition")["sample"].nunique())
df_bio.head(3)

## 4. Time-series plots

One line per condition; shaded band = 95 % bootstrapped CI across biological replicates.

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5))
if len(METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, METRICS):
    ic.time_series(df_bio, y=metric, time_col="elapsed_h", ax=ax)
    ax.set_xlabel("Time (h)")
    ax.set_title(metric)
plt.tight_layout()